# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/pile_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d08_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # pile train
    shutil.copy(glob.glob("/kaggle/input/**/d08_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # pile test
    print("Kaggle: pile data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: pile data copied to ./drive/MyDrive/Regression/pile_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795335713.360493      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795335732.459307      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (pile_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['F1', 'F2', 'Zv1', 'Zv2', 'phi', 'Th', 'L']
TARGET_LABEL = 'Pile response (Total)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/pile_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/pile_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (81, 8)
First 5 rows of training data:
      F1    F2   Zv1   Zv2  phi     Th   L  Total
0  4.80  1.21  4.90 -1.87  0.5  0.125  32   5.55
1  4.34  1.09  4.36 -1.95  0.5  0.125  45   6.07
2  4.71  0.81  5.10 -1.49  0.5  0.125  42   6.06
3  5.55  1.04  5.30 -1.46  0.5  0.125  38   5.94
4  3.08  0.95  2.82 -1.60  0.4  0.097  36   3.88

Shape of test data: (24, 8)
First 5 rows of test data:
      F1    F2   Zv1   Zv2  phi     Th   L  Total
0  4.69  1.06  4.70 -1.82  0.5  0.125  42   6.12
1  3.52  0.64  3.55 -1.03  0.4  0.097  45   4.26
2  3.70  0.49  3.43 -1.13  0.5  0.100  44   4.78
3  4.17  1.45  4.44 -0.21  0.5  0.125  16   5.78
4  4.33  2.04  4.75 -1.66  0.5  0.125  32   5.84


In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (81, 7)
Shape of y_train: (81,)
Shape of X_test: (24, 7)
Shape of y_test: (24,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[ 0.006145   -0.12114279  0.17814132 -0.04043083 -0.57071458  0.40496686
  -0.73776147]
 [-0.37545934 -0.27898568 -0.27593219 -0.10402107 -0.57071458  0.40496686
   0.23089542]
 [-0.06851672 -0.64728574  0.3463167   0.2616228  -0.57071458  0.40496686
   0.00735922]
 [ 0.62832598 -0.34475355  0.51449207  0.28546914 -0.57071458  0.40496686
  -0.29068906]
 [-1.42072338 -0.46313571 -1.57088258  0.17418622 -2.53785844 -2.27277321
  -0.4397132 ]]

First five rows of normalized X_test:
[[-8.51082130e-02 -3.18446397e-01  9.96594816e-03 -6.86931582e-04
  -5.70714577e-01  4.04966863e-01  7.35921670e-03]
 [-1.05571054e+00 -8.70896496e-01 -9.57042459e-01  6.27266667e-01
  -2.53785844e+00 -2.27277321e+00  2.30895424e-01]
 [-9.06387106e-01 -1.06820010e+00 -1.05794768e+00  5.47778870e-01
  -5.70714577e-01 -1.98587249e+00  1.56383355e-01]
 [-5.16487026e-01  1.94542980e-01 -2.08662040e-01  1.27906661e+00
  -5.70714577e-01  4.04966863e-01 -1.92995458e+00]
 [-3.83

# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
# FIX Q2: quantile crossing. Each quantile level is a separate model, so for some rows the
# predictions can come out in the wrong order (e.g. q0.10 < q0.05). Before any metric we sort
# each row's quantile predictions (monotone rearrangement, Chernozhukov et al. 2010) and record
# how many rows crossed, for the Excel report.
quantile_crossing_report = {}

def rearrange_quantiles(predictions_df, tag=None):
    q_cols = sorted(c for c in predictions_df.columns if c != 'Actual')
    values = predictions_df[q_cols].to_numpy(dtype=float)
    crossed = (np.diff(values, axis=1) < 0).any(axis=1)
    if tag is not None:
        quantile_crossing_report[tag] = {'rows': int(len(values)), 'rows_crossing': int(crossed.sum()),
                                         'pct_crossing': round(100.0 * crossed.mean(), 2)}
    out = predictions_df.copy()
    out[q_cols] = np.sort(values, axis=1)
    return out


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def quantile_regression_print(predictions_df, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"

    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title(f'Actual vs. Predicted Quantile {TARGET_LABEL}')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title(f'Actual {TARGET_LABEL} with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluation_features_print(predictions_df, X_test, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=feature_names)
    X_test.columns = feature_names

    # Get quantile columns from predictions_df except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Where to start images

    for feature_to_plot in feature_names:
        feature_values = X_test[feature_to_plot]
        plot_df = predictions_df.copy()
        plot_df[feature_to_plot] = feature_values.values

        # Sort by current feature to plot lines well
        plot_df = plot_df.sort_values(by=feature_to_plot)

        # Begin Matplotlib plot
        plt.figure(figsize=(6, 4))
        # Plot each quantile as a line
        for quantile in quantile_columns:
            plt.plot(plot_df[feature_to_plot], plot_df[quantile], label=f"{quantile}")

        # Plot actual values as scatter
        plt.scatter(plot_df[feature_to_plot], plot_df['Actual'], color='black', s=12, label=f"Actual {TARGET_LABEL}")

        plt.title(f"Predicted Quantiles vs. {feature_to_plot}")
        plt.xlabel(feature_to_plot)
        plt.ylabel(f"{TARGET_LABEL}")
        plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
        plt.tight_layout()

        # Save figure to a BytesIO stream
        img_bytes = BytesIO()
        img_bytes
        plt.savefig(img_bytes, format="png")
        plt.close()
        img_bytes.seek(0)

        img = Image(img_bytes)
        img.anchor = f"A{image_row}"
        ws.add_image(img)

        image_row += 20  # Adjust as you like

    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2

   # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(_ensure_excel_file(excel_file_path))

    # Create unique sheet names
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    # Create separate sheets for plots and values
    plots_sheet_title = get_unique_sheet_name(wb, 'Plots')
    values_sheet_title = get_unique_sheet_name(wb, 'Values')

    ws_plots = wb.create_sheet(title=plots_sheet_title)
    ws_values = wb.create_sheet(title=values_sheet_title)
    # FIX Q2: how often the raw quantile predictions crossed (before sorting)
    _cr = quantile_crossing_report.get(excel_file_path, {})
    ws_values.append(['Rows with quantile crossing (before sorting)', _cr.get('rows_crossing'), 'of', _cr.get('rows'), 'rows', f"{_cr.get('pct_crossing')}%"])
    print(f"Quantile crossing before sorting: {_cr.get('rows_crossing')} of {_cr.get('rows')} rows ({_cr.get('pct_crossing')}%)")

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the plots worksheet
    ws_plots.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        # Write interval length information to the values worksheet
        ws_values.append([f'Coverage level: {c*100:.0f}%', f'Interval: [{lower_quantile}, {upper_quantile}]', f'Average Interval Length: {avg_interval_length:.4f}'])

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)/2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    # Write average NLL to the values worksheet
    ws_values.append(['Average Negative Log-Likelihood (NLL)', average_nll])

    # Write NLL values to the values worksheet
    ws_values.append(['NLL Values'])
    for value in nll:
        ws_values.append([value])

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    # FIX Q4: CRPS from a set of quantiles = 2 * integral of the pinball loss over tau.
    # Each quantile level gets a weight equal to the share of [0, 1] it represents
    # (levels are unevenly spaced). Was: ps.crps_ensemble, which treats the 13 quantiles as
    # 13 equally likely samples.
    _taus = np.array(quantiles, dtype=float)
    _edges = np.concatenate([[0.0], (_taus[1:] + _taus[:-1]) / 2.0, [1.0]])
    _w = np.diff(_edges)
    _err = np.asarray(y_true, dtype=float)[:, None] - ensemble_predictions
    _pinball = np.maximum(_taus * _err, (_taus - 1.0) * _err)
    crps_values = 2.0 * (_pinball * _w).sum(axis=1)

    # Compute average CRPS
    average_crps = crps_values.mean()
    # Write average CRPS to the values worksheet
    ws_values.append(['Average Continuous Ranked Probability Score (CRPS)', average_crps])

    # Write CRPS values to the values worksheet
    ws_values.append(['CRPS Values'])
    for value in crps_values:
        ws_values.append([value])

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[3])

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    wb.save(_ensure_parent_dir(excel_file_path))

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    _ensure_parent_dir(image_path)
    plt.savefig(_ensure_parent_dir(image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_quantile_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation (not plotted, just for potential diagnostics)
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Matplotlib
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(predictions_df))
    # Actual Values
    ax.plot(x, actual_values, label='Actual', color='black', linewidth=2)

    # Mean Prediction
    ax.plot(x, mean_prediction, label='Mean Prediction', color='darkorange', linewidth=2)

    # Weighted Prediction Interval (blue fill)
    ax.fill_between(x, lower_bound, upper_bound, color='blue', alpha=0.3, label='Weighted Interval')
    # Optionally, plot the bounds as lines if you want:
    # ax.plot(x, lower_bound, color='blue', linewidth=1, linestyle='--')
    # ax.plot(x, upper_bound, color='blue', linewidth=1, linestyle='--')

    # Naive Prediction Interval (red fill)
    ax.fill_between(x, naive_lower_bound, naive_upper_bound, color='red', alpha=0.3, label='Naive Interval')
    # Optionally, plot the bounds as lines:
    # ax.plot(x, naive_lower_bound, color='red', linewidth=1, linestyle='--')
    # ax.plot(x, naive_upper_bound, color='red', linewidth=1, linestyle='--')

    ax.set_title('Quantile Prediction Intervals')
    ax.set_xlabel('Sample Number')
    ax.set_ylabel(f'{TARGET_LABEL}')
    ax.grid(False)
    ax.set_facecolor('white')

    # Place the legend outside the plot
    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), borderaxespad=0.0)

    plt.tight_layout(rect=[0,0,0.80,1.0])  # Make space for legend

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Add image to Excel
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="plot_quantile_intervals_to_excel")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_quantile_intervals_to_excel"

    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)
    ws.column_dimensions['A'].width = 30
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/pile_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshjangra/pile-tuned-params-v4/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 0.4182001959076784,
  'cv_rmse': 0.6466840000399564,
  'best_params': {'n_estimators': 300,
   'criterion': 'poisson',
   'max_depth': 20,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.3,
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.0},
  'test_mse': 0.1936734054166643,
  'test_rmse': np.float64(0.4400834073407725),
  'test_corr_coef': np.float64(0.9567730581463291),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 0.4131955900709805,
  'cv_rmse': 0.6428029169745424,
  'best_params': {'n_estimators': 300,
   'criterion': 'friedman_mse',
   'max_depth': None,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.3,
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.01,
   'n_

# **Quantile Regression with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  5.416515  5.532958  5.580026  5.639797  5.661070  5.708196  5.804273   
1  3.479340  3.618870  3.621726  3.124342  3.181224  3.470813  3.826506   
2  3.471142  3.613663  3.675100  3.267666  3.230597  3.423615  3.896266   
3  5.493280  5.582297  5.691177  5.616504  5.600414  5.605414  5.611938   
4  5.418842  5.609124  5.656727  5.600992  5.681168  5.705932  5.800535   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  5.869898  5.926781  5.993875  6.020839  5.987198  6.030840    6.12  
1  4.520462  4.745034  5.570195  5.549571  5.654379  5.751656    4.26  
2  4.436177  4.755539  5.597687  5.537905  5.691672  5.666775    4.78  
3  5.665609  5.590391  5.625637  5.618371  5.660873  5.715204    5.78  
4  5.852198  5.769128  5.781538  5.762640  5.771114  5.894907    5.84  


In [21]:
quantile_regression_print(predictions_LGBM_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

First few actual values: [6.12 4.26 4.78 5.78 5.84]
Coverage of 90% prediction interval: 58.33%


In [22]:
evaluation_features_print(predictions_LGBM_df,x_test,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

In [23]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/LightGBM.xlsx", model_name='LightGBM')

Quantile crossing before sorting: 24 of 24 rows (100.0%)


{'empirical_quantiles': [np.float64(0.041666666666666664),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.375),
  np.float64(0.4583333333333333),
  np.float64(0.5416666666666666),
  np.float64(0.5416666666666666),
  np.float64(0.625),
  np.float64(0.625)],
 'average_interval_lengths': [np.float64(0.513222927752809),
  np.float64(0.7599514258347883),
  np.float64(1.1191709210836585)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.793733875843619),
 'average_crps': np.float64(0.25975879947171704),
 'nll_values': array([ 0.67016583,  0.84140999,  1.32645248,  1.15961504, -0.87117591,
        -0.87641602,  0.33348111,  2.58042379,  0.45313021, -0.42707604,
        -0.85995377, -0.70153295,  2.14176164,  4.91743412, -0.05693919,
        -0.03524292,  3.26874223, -0.35885346,  0.69895632,  0.81636

In [24]:
calibrate_and_plot_intervals(predictions_LGBM_df, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

The empirical coverage before calibration is: 50.00%
The empirical coverage after calibration is: 100.00%


In [25]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_LGBM_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

# **Quantile Regression with XGBoost**

In [26]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_paramst.copy()  # FIX Q1: was best_params (= LightGBM's settings from the cell above)

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  5.482893  5.721023  5.747579  5.779704  5.460186  5.890114  5.892218   
1  3.108167  3.133558  3.126244  3.302843  3.321558  3.358237  3.684932   
2  3.060684  2.970085  3.126387  3.290060  3.356553  3.143436  3.388966   
3  5.571816  5.594025  5.610260  5.696156  5.645147  5.716146  5.734402   
4  5.450380  5.702892  5.692569  5.700668  5.740442  5.618315  5.696072   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  5.792463  5.867325  5.899883  5.897031  5.912476  5.963117    6.12  
1  3.350401  3.715098  4.474362  4.349173  4.876101  4.970625    4.26  
2  3.180673  3.261129  4.149108  3.887193  4.157712  5.053194    4.78  
3  5.575781  5.760124  5.670723  5.615633  5.683966  5.792861    5.78  
4  5.709063  5.737885  5.669305  5.699203  5.681314  5.802940    5.84  


In [27]:
quantile_regression_print(predictions_XGB_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

First few actual values: [6.12 4.26 4.78 5.78 5.84]
Coverage of 90% prediction interval: 58.33%


In [28]:
evaluation_features_print(predictions_XGB_df,x_test,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

In [29]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/XGBoost.xlsx", model_name='XGBoost')

Quantile crossing before sorting: 24 of 24 rows (100.0%)


{'empirical_quantiles': [np.float64(0.08333333333333333),
  np.float64(0.08333333333333333),
  np.float64(0.08333333333333333),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.25),
  np.float64(0.25),
  np.float64(0.2916666666666667),
  np.float64(0.375),
  np.float64(0.4166666666666667),
  np.float64(0.4583333333333333),
  np.float64(0.6666666666666666)],
 'average_interval_lengths': [np.float64(0.2918618718783061),
  np.float64(0.5503771205743154),
  np.float64(0.86541215578715)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(1.1868417336381436),
 'average_crps': np.float64(0.3301399212280908),
 'nll_values': array([ 0.40665362,  1.61855534,  3.23020614, -0.45878691, -0.45140484,
        -1.1222625 , -0.13303587,  4.8248739 , -0.4825772 ,  2.01979469,
        -1.15753593, -0.89602393,  6.28187843,  6.64599458,  0.32035428,
        -0.71792067,  1.54412577, -0.40606911,  1.28635004,  2.35886475,


In [30]:
calibrate_and_plot_intervals(predictions_XGB_df, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

The empirical coverage before calibration is: 66.67%
The empirical coverage after calibration is: 100.00%


In [31]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_XGB_df, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

# **Quantile Regression with GPBoost**

In [32]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  5.557479  5.708309  5.401857  5.682226  5.767388  5.672527  5.758426   
1  3.094191  3.146003  3.048008  3.249061  3.277111  3.395373  3.508932   
2  3.144079  3.241188  3.127914  3.251476  3.132543  3.433424  3.530202   
3  5.826890  5.508118  5.358047  5.484860  5.469225  5.679261  5.762877   
4  5.689803  5.772815  5.734479  5.928282  5.886001  5.818382  5.889171   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  5.864612  5.757331  5.744427  5.896071  5.960798  6.020318    6.12  
1  3.668990  4.157078  5.140419  5.552668  5.802304  5.669331    4.26  
2  3.573653  4.114766  5.036576  5.459884  5.727903  5.682991    4.78  
3  5.818214  5.654522  5.614872  5.574269  5.660267  5.857221    5.78  
4  5.818152  5.808256  5.852390  5.779391  5.773914  5.863342    5.84  


In [33]:
quantile_regression_print(predictions_GPBoost_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

First few actual values: [6.12 4.26 4.78 5.78 5.84]
Coverage of 90% prediction interval: 62.50%


In [34]:
evaluation_features_print(predictions_GPBoost_df,x_test,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

In [35]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/GPBoost.xlsx", model_name='GPBoost')

Quantile crossing before sorting: 24 of 24 rows (100.0%)


{'empirical_quantiles': [np.float64(0.125),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.3333333333333333),
  np.float64(0.4583333333333333),
  np.float64(0.5833333333333334),
  np.float64(0.7083333333333334),
  np.float64(0.75),
  np.float64(0.75)],
 'average_interval_lengths': [np.float64(0.48489510277521153),
  np.float64(0.7860255168874423),
  np.float64(1.0546126505474152)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.9250069279900757),
 'average_crps': np.float64(0.2705425416940892),
 'nll_values': array([ 1.10834007,  1.14366869,  1.93396673, -0.62474657, -1.65992012,
        -0.50997615,  0.47133166,  2.95020773,  0.65587718, -0.71256926,
        -0.97970972, -0.99159213,  3.13475755,  2.98277514, -0.41445281,
         0.26487011,  4.71583914, -0.73635102,  0.97583582,  4.00982514,

In [36]:
calibrate_and_plot_intervals(predictions_GPBoost_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

The empirical coverage before calibration is: 66.67%
The empirical coverage after calibration is: 91.67%


In [37]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GPBoost_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

# **Quantile Regression with NGBoost**

In [38]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)

    # Compute the quantile prediction per sample
    predictions = mu + z * sigma

    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=1.7359 val_loss=0.0000 scale=2.0000 norm=2.4749


[iter 100] loss=-1.9095 val_loss=0.0000 scale=1.0000 norm=0.1983


[iter 200] loss=-3.3133 val_loss=0.0000 scale=0.5000 norm=0.1083


[iter 300] loss=-4.2158 val_loss=0.0000 scale=0.5000 norm=0.0934


[iter 400] loss=-4.9786 val_loss=0.0000 scale=0.2500 norm=0.0513


       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  5.862383  5.863034  5.863473  5.863822  5.864390  5.864875  5.865329   
1  3.766763  3.766947  3.767071  3.767170  3.767330  3.767468  3.767596   
2  3.261190  3.261365  3.261483  3.261576  3.261729  3.261859  3.261981   
3  5.820536  5.821230  5.821698  5.822070  5.822675  5.823193  5.823677   
4  5.759687  5.760288  5.760693  5.761016  5.761540  5.761988  5.762407   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  5.865782  5.866268  5.866836  5.867185  5.867624  5.868274    6.12  
1  3.767724  3.767861  3.768022  3.768121  3.768245  3.768429    4.26  
2  3.262103  3.262233  3.262385  3.262479  3.262597  3.262771    4.78  
3  5.824160  5.824678  5.825284  5.825656  5.826124  5.826817    5.78  
4  5.762826  5.763275  5.763799  5.764121  5.764527  5.765128    5.84  


In [39]:
quantile_regression_print(predictions_NGB_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

First few actual values: [6.12 4.26 4.78 5.78 5.84]
Coverage of 90% prediction interval: 0.00%


In [40]:
evaluation_features_print(predictions_NGB_df,x_test,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

In [41]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/NGBoost.xlsx", model_name='NGBoost')

Quantile crossing before sorting: 0 of 24 rows (0.0%)


{'empirical_quantiles': [np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667)],
 'average_interval_lengths': [np.float64(0.0015855075563276182),
  np.float64(0.0024142804505636883),
  np.float64(0.0030986954113703855)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(967574.5765726516),
 'average_crps': np.float64(0.4603014414583133),
 'nll_values': array([ 1.01081402e+04,  4.73148311e+05,  4.98884662e+06,  2.56264257e+02,
         1.09516036e+03,  8.18013567e+03,  6.45762146e+05,  1.65925261e+05,
         1.31403983e+03,  8.98671713e+03, -1.74807425e+00,  2.37589038e+04,
         3.26694241

In [42]:
calibrate_and_plot_intervals(predictions_NGB_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

The empirical coverage before calibration is: 0.00%
The empirical coverage after calibration is: 100.00%


In [43]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_NGB_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

# **Quantile Regression with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GBR_df = pd.DataFrame()

# Custom quantile loss function
def quantile_loss(q, y_true, y_pred):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

# Train a model for each quantile and make predictions using GradientBoostingRegressor
for q in quantiles:

    params = best_params.copy()

    params.update({
        'loss': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    # Initialize the model with quantile-specific settings
    model = GradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GBR_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  5.457548  5.792222  5.648513  5.669668  6.017354  5.947635  5.668948   
1  4.031053  3.633701  3.576891  3.222406  3.351269  2.918727  3.085589   
2  3.990004  3.178020  3.446724  2.994045  3.570593  2.871946  3.452021   
3  5.547957  5.713420  6.496926  5.812867  6.085598  5.688588  5.639734   
4  5.866081  5.544938  5.828515  5.870583  5.785969  6.109320  5.804677   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  5.625660  5.687494  5.828288  5.770925  5.622788  5.866566    6.12  
1  4.335093  2.980563  2.962757  3.665084  2.808983  3.945394    4.26  
2  3.860985  3.044936  3.147324  3.364413  4.007403  4.136617    4.78  
3  5.704556  5.712037  5.254884  5.531199  5.596764  4.926337    5.78  
4  5.796009  5.716932  5.657050  5.738419  5.650099  5.921355    5.84  


In [45]:
quantile_regression_print(predictions_GBR_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

First few actual values: [6.12 4.26 4.78 5.78 5.84]
Coverage of 90% prediction interval: 58.33%


In [46]:
evaluation_features_print(predictions_GBR_df,x_test,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

In [47]:
evaluate_uncertainity_matrix(predictions_GBR_df, quantiles,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx", model_name='GBM')

Quantile crossing before sorting: 24 of 24 rows (100.0%)


{'empirical_quantiles': [np.float64(0.041666666666666664),
  np.float64(0.08333333333333333),
  np.float64(0.125),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.25),
  np.float64(0.25),
  np.float64(0.3333333333333333),
  np.float64(0.3333333333333333),
  np.float64(0.375),
  np.float64(0.4166666666666667),
  np.float64(0.5),
  np.float64(0.625)],
 'average_interval_lengths': [np.float64(0.30439939120530374),
  np.float64(0.6258378477498295),
  np.float64(0.9801452947124166)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(1.2376625284824),
 'average_crps': np.float64(0.3328076213507262),
 'nll_values': array([ 2.37790976,  2.06945912,  5.97703406,  0.1979206 , -0.81100972,
        -0.67974804,  0.12005118,  3.23832849,  0.09559162,  0.72890465,
        -0.992424  , -0.42974578,  0.47037489,  7.51966917,  0.96989776,
         0.07554461,  0.85834205, -0.94117771,  1.03848543,  3.44796685,
         2.11403116, -0.03258696, -0.34566

In [48]:
calibrate_and_plot_intervals(predictions_GBR_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

The empirical coverage before calibration is: 83.33%
The empirical coverage after calibration is: 100.00%


In [49]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GBR_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

# **Quantile Regression with CatBoost**

In [50]:
# Get the best hyperparameters for CatBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_CatBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'loss_function', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'loss_function': 'Quantile:alpha={}'.format(q),
        'random_seed': 42
    })

    # Initialize the model with the updated parameters
    model = CatBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train, verbose=0)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_CatBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_CatBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_CatBoost_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  5.581769  5.641036  5.620360  5.661996  5.740367  5.727333  5.745458   
1  3.832641  3.844137  3.899071  4.645793  4.601394  4.365405  4.099220   
2  3.793470  3.914051  3.576313  4.200944  4.090326  3.993686  3.504842   
3  5.431178  5.449985  5.499055  5.582556  5.675932  5.673472  5.694289   
4  5.633740  5.631962  5.506313  5.615815  5.657671  5.715621  5.697841   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  5.858612  5.898972  5.872160  5.942538  5.939816  6.027957    6.12  
1  4.375690  4.379479  4.118106  3.546029  4.377855  5.140693    4.26  
2  4.097960  4.236703  4.173520  4.244441  4.823459  4.908530    4.78  
3  5.743962  5.567743  5.718587  5.653908  5.584195  5.595951    5.78  
4  5.697638  5.735008  5.707291  5.756885  5.766337  5.836644    5.84  


In [51]:
quantile_regression_print(predictions_CatBoost_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

First few actual values: [6.12 4.26 4.78 5.78 5.84]
Coverage of 90% prediction interval: 50.00%


In [52]:
evaluation_features_print(predictions_CatBoost_df,x_test,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

In [53]:
evaluate_uncertainity_matrix(predictions_CatBoost_df, quantiles,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/CatBoost.xlsx", model_name='CatBoost')

Quantile crossing before sorting: 24 of 24 rows (100.0%)


{'empirical_quantiles': [np.float64(0.08333333333333333),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.25),
  np.float64(0.25),
  np.float64(0.2916666666666667),
  np.float64(0.375),
  np.float64(0.375),
  np.float64(0.4166666666666667),
  np.float64(0.4583333333333333),
  np.float64(0.4583333333333333),
  np.float64(0.5416666666666666),
  np.float64(0.5833333333333334)],
 'average_interval_lengths': [np.float64(0.2586173684762599),
  np.float64(0.5080770449459364),
  np.float64(0.7260852410717828)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.507412951139611),
 'average_crps': np.float64(0.25496383850531745),
 'nll_values': array([ 2.73399174,  0.21844444,  1.34474924,  0.43943276, -0.37735904,
        -1.12193724,  0.38730213,  5.46476459,  1.26878053,  0.67951101,
        -0.98312175, -0.13818302,  0.54597377, 30.83864568, -0.26146153,
        -0.25015421,  5.88021404, -0.68889813, -0.21955029,  9.35341299,
         1.990

In [54]:
calibrate_and_plot_intervals(predictions_CatBoost_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

The empirical coverage before calibration is: 50.00%
The empirical coverage after calibration is: 100.00%


In [55]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_CatBoost_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

# **Quantile Regression with HistGradientBoosting**

In [56]:

# Assume best_params is a dictionary containing the best parameters for HistGradientBoostingRegressor
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_HGBR_df = pd.DataFrame()

# Train a model for each quantile and make predictions using HistGradientBoostingRegressor
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'loss': 'quantile',
        'quantile': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = HistGradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_HGBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_HGBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_HGBR_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  5.352570  5.679471  5.657194  5.760710  5.657903  5.595060  5.627321   
1  3.021243  3.123570  3.225307  2.859188  3.601735  3.268053  4.234698   
2  2.901815  3.101494  3.107042  2.817843  3.601577  3.304852  4.199985   
3  5.352570  5.579887  5.586270  5.696889  5.605992  5.556959  5.440143   
4  5.352570  5.588486  5.635752  5.638897  5.814633  5.717670  5.839116   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  5.693263  5.891605  5.864655  5.936677  6.023809  6.783928    6.12  
1  5.324576  5.158622  5.634541  5.616947  5.838986  6.783928    4.26  
2  5.312056  5.239689  5.629654  5.616947  5.811823  6.783928    4.78  
3  5.510939  5.597348  5.667902  5.670392  5.757007  6.783928    5.78  
4  5.835078  5.781612  5.720915  5.788030  5.817665  6.783928    5.84  


In [57]:
quantile_regression_print(predictions_HGBR_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/HGBM.xlsx")

First few actual values: [6.12 4.26 4.78 5.78 5.84]
Coverage of 90% prediction interval: 91.67%


In [58]:
evaluation_features_print(predictions_HGBR_df,x_test,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/HGBM.xlsx")

In [59]:
evaluate_uncertainity_matrix(predictions_HGBR_df, quantiles,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/HGBM.xlsx", model_name='HGBM')

Quantile crossing before sorting: 24 of 24 rows (100.0%)


{'empirical_quantiles': [np.float64(0.0),
  np.float64(0.125),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.20833333333333334),
  np.float64(0.25),
  np.float64(0.4166666666666667),
  np.float64(0.5),
  np.float64(0.5833333333333334),
  np.float64(0.625),
  np.float64(0.7083333333333334),
  np.float64(0.9166666666666666)],
 'average_interval_lengths': [np.float64(0.8840036589377824),
  np.float64(1.200661128709739),
  np.float64(2.3373889617954924)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.7251472867627635),
 'average_crps': np.float64(0.2858502603918532),
 'nll_values': array([0.56772296, 1.09566497, 1.22164761, 0.17487604, 0.09576801,
        0.13612678, 0.89277413, 1.12427138, 0.85688979, 0.20326146,
        0.11218519, 0.09082927, 1.02224432, 1.07391151, 0.78674223,
        0.88591815, 0.40339351, 0.12746598, 1.27161921, 1.24344663,
        1.27340652, 0.90794358, 0.16197676, 1.6734

In [60]:
calibrate_and_plot_intervals(predictions_HGBR_df, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/HGBM.xlsx")

The empirical coverage before calibration is: 100.00%
The empirical coverage after calibration is: 100.00%


In [61]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_HGBR_df, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/HGBM.xlsx")

# **Quantile Regression with TabNet**

In [62]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')

# Quantiles
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Storage
models_quantile = {}
predictions_TabNet_df = pd.DataFrame()

# Convert data to numpy (TabNet requirement)
X_train_np = np.array(X_train)
y_train_np = np.array(y_train).reshape(-1, 1)
X_test_np  = np.array(X_test)

# Quantile loss (pinball loss)
def quantile_loss(q):
    def loss(y_pred, y_true):
        error = y_true - y_pred
        return torch.mean(torch.maximum(q * error, (q - 1) * error))
    return loss

# FIX Q3: early stopping on a 20% validation split of TRAIN, scored with the pinball loss of
# the quantile being trained (was: eval_set = the training data itself, scored with MSE,
# so early stopping never triggered).
from pytorch_tabnet.metrics import Metric
X_fit_np, X_val_np, y_fit_np, y_val_np = train_test_split(X_train_np, y_train_np, test_size=0.2, random_state=42)

def pinball_metric(q):
    class Pinball(Metric):
        def __init__(self):
            self._name = f'pinball_{q}'
            self._maximize = False
        def __call__(self, y_true, y_score, weights=None):  # FIX B8: pytorch-tabnet2 also passes sample weights
            e = np.asarray(y_true, dtype=float) - np.asarray(y_score, dtype=float)
            return float(np.mean(np.maximum(q * e, (q - 1) * e)))
    return Pinball

# Train one TabNet per quantile
for q in quantiles:
    model = TabNetRegressor(
        **best_params,
    )

    model.fit(
        X_fit_np,
        y_fit_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_metric=[pinball_metric(q)],
        loss_fn=quantile_loss(q),
        max_epochs=200,
        patience=30,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    # Store model
    models_quantile[q] = model

    # Predict
    preds = model.predict(X_test_np).ravel()
    predictions_TabNet_df[q] = preds

# Add actual values
predictions_TabNet_df['Actual'] = y_test.values.ravel()

print(predictions_TabNet_df.head())


Early stopping occurred at epoch 34 with best_epoch = 4 and best_val_0_pinball_0.05 = 0.19476



Early stopping occurred at epoch 73 with best_epoch = 43 and best_val_0_pinball_0.1 = 0.17745



Early stopping occurred at epoch 44 with best_epoch = 14 and best_val_0_pinball_0.15 = 0.36206



Early stopping occurred at epoch 41 with best_epoch = 11 and best_val_0_pinball_0.2 = 0.48954



Early stopping occurred at epoch 90 with best_epoch = 60 and best_val_0_pinball_0.3 = 0.49821



Early stopping occurred at epoch 96 with best_epoch = 66 and best_val_0_pinball_0.4 = 0.40975


... (3 outputs left out)


Early stopping occurred at epoch 169 with best_epoch = 139 and best_val_0_pinball_0.8 = 0.23993



Early stopping occurred at epoch 97 with best_epoch = 67 and best_val_0_pinball_0.85 = 0.35623


Stop training because you reached max_epochs = 200 with best_epoch = 177 and best_val_0_pinball_0.9 = 0.1465



Early stopping occurred at epoch 82 with best_epoch = 52 and best_val_0_pinball_0.95 = 0.14994
       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  0.663719  4.000045  2.782529  2.041323  4.968060  5.298853  5.672302   
1  2.482443  4.044893  5.724821  6.889705  6.161522  5.384425  8.534126   
2  0.422984  4.580626  3.321587  3.126853  4.921974  4.422547  5.092546   
3  0.206337  3.984526  1.954980  1.959667  6.381913  5.793514  5.203405   
4  1.016601  5.724045  4.106301  3.165800  6.132153  6.135130  5.823965   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  6.039588  5.827899  5.800210  5.372940  6.210894  6.611054    6.12  
1  3.923705  6.198089  6.962796  5.794713  9.539295  9.059477    4.26  
2  2.862099  5.384637  5.285654  4.182600  5.984306  4.427339    4.78  
3  4.503495  4.877183  5.613200  4.865074  5.377439  5.410110    5.78  
4  5.672341  5.953947  5.817889  5.631252  6.339375  7.418444    5.84  


In [63]:
quantile_regression_print(predictions_TabNet_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/TabNet.xlsx")

First few actual values: [6.12 4.26 4.78 5.78 5.84]
Coverage of 90% prediction interval: 87.50%


In [64]:
evaluation_features_print(predictions_TabNet_df,x_test,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/TabNet.xlsx")

In [65]:
evaluate_uncertainity_matrix(predictions_TabNet_df, quantiles, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/TabNet.xlsx", model_name='TabNet')

Quantile crossing before sorting: 24 of 24 rows (100.0%)


{'empirical_quantiles': [np.float64(0.0),
  np.float64(0.0),
  np.float64(0.041666666666666664),
  np.float64(0.125),
  np.float64(0.16666666666666666),
  np.float64(0.2916666666666667),
  np.float64(0.3333333333333333),
  np.float64(0.375),
  np.float64(0.4583333333333333),
  np.float64(0.5),
  np.float64(0.5416666666666666),
  np.float64(0.7083333333333334),
  np.float64(0.875)],
 'average_interval_lengths': [np.float64(1.5135249694188435),
  np.float64(3.6147416631380715),
  np.float64(5.909981531401475)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(1.655100964385129),
 'average_crps': np.float64(0.6267437968539694),
 'nll_values': array([1.59646219, 2.07502375, 1.46573507, 1.66438772, 1.58479043,
        1.48161703, 1.66911233, 1.55501378, 1.80986117, 1.64864979,
        1.51549367, 1.64485465, 1.88487298, 1.75346534, 1.53717143,
        1.52359498, 1.60966403, 1.32812486, 2.22961372, 1.77081487,
        2.01860531, 1.69537963, 1.36058252, 1.29953188]),
 'crps_v

In [66]:
calibrate_and_plot_intervals(predictions_TabNet_df, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/TabNet.xlsx")

The empirical coverage before calibration is: 83.33%
The empirical coverage after calibration is: 75.00%


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_TabNet_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/TabNet.xlsx")

# **Quantile Regression with PGBM**

In [68]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [69]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU
Estimator 0/300, Train metric: 1.2878


Estimator 1/300, Train metric: 1.1895
Estimator 2/300, Train metric: 1.1113
Estimator 3/300, Train metric: 1.0241
Estimator 4/300, Train metric: 0.9480
Estimator 5/300, Train metric: 0.8808
Estimator 6/300, Train metric: 0.8264
Estimator 7/300, Train metric: 0.7738
Estimator 8/300, Train metric: 0.7184
Estimator 9/300, Train metric: 0.6700
Estimator 10/300, Train metric: 0.6375
Estimator 11/300, Train metric: 0.5968
Estimator 12/300, Train metric: 0.5604
Estimator 13/300, Train metric: 0.5311
Estimator 14/300, Train metric: 0.5040
Estimator 15/300, Train metric: 0.4810
Estimator 16/300, Train metric: 0.4570
Estimator 17/300, Train metric: 0.4311
Estimator 18/300, Train metric: 0.4174
Estimator 19/300, Train metric: 0.4061
Estimator 20/300, Train metric: 0.3897
Estimator 21/300, Train metric: 0.3722
Estimator 22/300, Train metric: 0.3613
Estimator 23/300, Train metric: 0.3421
Estimator 24/300, Train metric: 0.3281
Estimator 25/300, Train metric: 0.3154
Estimator 26/300, Train metric: 0.

Estimator 30/300, Train metric: 0.2784
Estimator 31/300, Train metric: 0.2742
Estimator 32/300, Train metric: 0.2692
Estimator 33/300, Train metric: 0.2635
Estimator 34/300, Train metric: 0.2571
Estimator 35/300, Train metric: 0.2541
Estimator 36/300, Train metric: 0.2461
Estimator 37/300, Train metric: 0.2396
Estimator 38/300, Train metric: 0.2347
Estimator 39/300, Train metric: 0.2311
Estimator 40/300, Train metric: 0.2257
Estimator 41/300, Train metric: 0.2239
Estimator 42/300, Train metric: 0.2179
Estimator 43/300, Train metric: 0.2173
Estimator 44/300, Train metric: 0.2123
Estimator 45/300, Train metric: 0.2115
Estimator 46/300, Train metric: 0.2070
Estimator 47/300, Train metric: 0.2071
Estimator 48/300, Train metric: 0.2049
Estimator 49/300, Train metric: 0.2035
Estimator 50/300, Train metric: 0.2035
Estimator 51/300, Train metric: 0.2035
Estimator 52/300, Train metric: 0.1999
Estimator 53/300, Train metric: 0.1999
Estimator 54/300, Train metric: 0.1999
Estimator 55/300, Train m

Estimator 175/300, Train metric: 0.1838
Estimator 176/300, Train metric: 0.1838
Estimator 177/300, Train metric: 0.1838
Estimator 178/300, Train metric: 0.1838
Estimator 179/300, Train metric: 0.1838
Estimator 180/300, Train metric: 0.1838
Estimator 181/300, Train metric: 0.1838
Estimator 182/300, Train metric: 0.1838
Estimator 183/300, Train metric: 0.1838
Estimator 184/300, Train metric: 0.1838
Estimator 185/300, Train metric: 0.1838
Estimator 186/300, Train metric: 0.1838
Estimator 187/300, Train metric: 0.1838
Estimator 188/300, Train metric: 0.1838
Estimator 189/300, Train metric: 0.1838
Estimator 190/300, Train metric: 0.1838
Estimator 191/300, Train metric: 0.1838
Estimator 192/300, Train metric: 0.1838
Estimator 193/300, Train metric: 0.1838
Estimator 194/300, Train metric: 0.1838
Estimator 195/300, Train metric: 0.1838
Estimator 196/300, Train metric: 0.1838
Estimator 197/300, Train metric: 0.1838
Estimator 198/300, Train metric: 0.1838
Estimator 199/300, Train metric: 0.1838


In [70]:
quantile_regression_print(predictions_PGBM_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/PGBM.xlsx")

First few actual values: [6.12 4.26 4.78 5.78 5.84]
Coverage of 90% prediction interval: 37.50%


In [71]:
evaluation_features_print(predictions_PGBM_df,x_test,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/PGBM.xlsx")

In [72]:
evaluate_uncertainity_matrix(predictions_PGBM_df, quantiles, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/PGBM.xlsx", model_name='PGBM')

Quantile crossing before sorting: 0 of 24 rows (0.0%)


{'empirical_quantiles': [np.float64(0.08333333333333333),
  np.float64(0.08333333333333333),
  np.float64(0.125),
  np.float64(0.20833333333333334),
  np.float64(0.25),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.2916666666666667),
  np.float64(0.3333333333333333),
  np.float64(0.375),
  np.float64(0.375),
  np.float64(0.375),
  np.float64(0.4583333333333333)],
 'average_interval_lengths': [np.float64(0.12177843848864238),
  np.float64(0.21046486496925354),
  np.float64(0.29553372661272687)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(14.894058989796909),
 'average_crps': np.float64(0.3295105468779802),
 'nll_values': array([ 6.34553247, 13.65609852, 79.73840923, -0.91480736, -0.69798449,
        -1.14353102, -1.44804489, 50.59147995,  4.30041011, 21.7360826 ,
        -1.51364542, -1.45865159, 20.99561589, 14.12996185,  3.86966446,
        -0.86585524,  3.31914111, -1.30342234, 24.77146429, 24.45346658,
         3.55727106, -0

In [73]:
calibrate_and_plot_intervals(predictions_PGBM_df, "./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/PGBM.xlsx")

The empirical coverage before calibration is: 50.00%
The empirical coverage after calibration is: 100.00%


In [74]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_PGBM_df,"./drive/MyDrive/Regression/pile_uncertainty_analysis/quantile_regression/PGBM.xlsx")